# Detecção e Classificação de Lesões Mamárias em Ultrassonografia com YOLO11

Detector YOLO11s (Ultralytics) para localização e classificação de lesões benignas e malignas em imagens de ultrassom do dataset BUSI. Pipeline reprodutível: deduplicação, divisão estratificada 70/15/15 com semente fixa e avaliação por `model.val()`.

**Autores:** Juan José Gouvêa Cardenas e Felipe Ramirez Pereira Botero.

*Pesquisa acadêmica / prova de conceito. Não é dispositivo médico e não substitui avaliação profissional.*

## 1. Instalação dos pacotes

- `ultralytics`: modelos YOLO (treino, validação, inferência), incluindo o YOLO11.
- `opencv-python-headless`: leitura/processamento de imagens e máscaras.
- `pyyaml`: leitura/escrita do arquivo de configuração do dataset.
- `tqdm`: barras de progresso.
- `scikit-learn`: divisão estratificada treino/validação/teste.

In [ ]:
!pip install -q ultralytics opencv-python-headless pyyaml tqdm scikit-learn

## 2. Importação das bibliotecas e ambiente

In [ ]:
import os
import glob
import json
import shutil
import hashlib
import platform
from collections import Counter

import yaml
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import ultralytics
import matplotlib.pyplot as plt

# Versões do ambiente (reprodutibilidade)
env_info = {'python': platform.python_version(), 'ultralytics': ultralytics.__version__}
try:
    import torch
    env_info['torch'] = torch.__version__
    env_info['cuda_available'] = torch.cuda.is_available()
    env_info['gpu'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
except Exception as e:
    env_info['torch'] = f'indisponível ({e})'
env_info['opencv'] = cv2.__version__
print('Ambiente de execução:')
for k, v in env_info.items():
    print(f'  {k:15s}: {v}')

## 3. Diretório base e semente

In [ ]:
base_dir = '/kaggle/working/Breast_Ultrasound_Project'
os.makedirs(base_dir, exist_ok=True)

SEED = 42
np.random.seed(SEED)

## 4. Localização do dataset BUSI

Localiza a pasta do BUSI sob `/kaggle/input/`. Se não encontrar, anexe o *Breast Ultrasound Images Dataset* em **Add Input**.

In [ ]:
def encontrar_busi():
    # 1) pasta canônica Dataset_BUSI_with_GT em qualquer profundidade
    for hit in glob.glob('/kaggle/input/**/Dataset_BUSI_with_GT', recursive=True):
        if os.path.isdir(hit):
            return hit
    # 2) fallback: qualquer pasta que contenha benign/ malignant/ normal
    for cls_dir in glob.glob('/kaggle/input/**/benign', recursive=True):
        root = os.path.dirname(cls_dir)
        if all(os.path.isdir(os.path.join(root, c)) for c in ('benign', 'malignant', 'normal')):
            return root
    return None

dataset_path = encontrar_busi()
assert dataset_path is not None, (
    'Dataset BUSI não encontrado em /kaggle/input/. No painel direito do Kaggle, '
    'clique em "Add Input" e adicione o "Breast Ultrasound Images Dataset" (aryashah2k). '
    'Pastas esperadas: benign/ malignant/ normal.'
)
print('Dataset BUSI encontrado em:', dataset_path)
for c in ('benign', 'malignant', 'normal'):
    n_png = len(glob.glob(os.path.join(dataset_path, c, '*.png')))
    print(f'  {c:10s}: {n_png} arquivos .png (imagens + máscaras)')

## 5. Catalogação das imagens e máscaras

In [ ]:
classes = ['benign', 'malignant', 'normal']
images_list = []

for cls in classes:
    cls_path = os.path.join(dataset_path, cls)
    for image_file in glob.glob(os.path.join(cls_path, '*.png')):
        if '_mask' in os.path.basename(image_file):
            continue  # é uma máscara, não a imagem
        stem = image_file[:-4]  # remove '.png'
        masks = sorted(glob.glob(stem + '_mask*.png'))  # _mask.png, _mask_1.png, ...
        images_list.append({'image_path': image_file, 'masks': masks, 'class': cls})

print('Total catalogado:', len(images_list))
print('Por classe:', dict(Counter(it['class'] for it in images_list)))
multi = sum(1 for it in images_list if len(it['masks']) > 1)
print(f'Imagens com múltiplas máscaras (multilesão): {multi}')

## 6. Deduplicação

Remoção de duplicatas exatas (hash MD5) antes da divisão, evitando vazamento entre os conjuntos.

In [ ]:
def file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

seen, deduped = set(), []
for it in tqdm(images_list, desc='Deduplicando'):
    h = file_hash(it['image_path'])
    if h in seen:
        continue
    seen.add(h)
    deduped.append(it)

removidas = len(images_list) - len(deduped)
print(f'Duplicatas exatas removidas: {removidas}')
print('Após deduplicação:', len(deduped))
print('Por classe:', dict(Counter(it['class'] for it in deduped)))
images_list = deduped

## 7. Divisão estratificada em treino / validação / teste (70 / 15 / 15)

In [ ]:
labels = [it['class'] for it in images_list]

train_items, temp_items = train_test_split(
    images_list, test_size=0.30, random_state=SEED, stratify=labels
)
temp_labels = [it['class'] for it in temp_items]
val_items, test_items = train_test_split(
    temp_items, test_size=0.50, random_state=SEED, stratify=temp_labels
)

split_counts = {}
for nome, subset in [('treino', train_items), ('val', val_items), ('teste', test_items)]:
    c = dict(Counter(it['class'] for it in subset))
    split_counts[nome] = {'total': len(subset), **c}
    print(f'{nome:7s}: {len(subset):3d}  ->  {c}')

## 8. Geração dos rótulos YOLO

Conversão das máscaras em caixas (uma por lesão). Classes: `0 = benigno`, `1 = maligno`. Imagens normais recebem rótulo vazio (exemplo negativo).

In [ ]:
class_mapping = {'benign': 0, 'malignant': 1}

def masks_to_yolo_lines(mask_paths, width, height, class_id):
    """Uma linha de rótulo YOLO por máscara não vazia (uma caixa por lesão)."""
    linhas = []
    for mp in mask_paths:
        mask = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue
        coords = cv2.findNonZero(mask)
        if coords is None:
            continue
        x, y, w, h = cv2.boundingRect(coords)
        x_c, y_c = (x + w / 2) / width, (y + h / 2) / height
        linhas.append(f'{class_id} {x_c:.6f} {y_c:.6f} {w / width:.6f} {h / height:.6f}')
    return linhas

def materializar(subset, split):
    img_dir = os.path.join(base_dir, 'images', split)
    lbl_dir = os.path.join(base_dir, 'labels', split)
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)
    n_lesoes = 0
    for it in subset:
        img = cv2.imread(it['image_path'])
        if img is None:
            continue
        height, width = img.shape[:2]
        linhas = []
        if it['class'] != 'normal':
            linhas = masks_to_yolo_lines(it['masks'], width, height, class_mapping[it['class']])
            n_lesoes += len(linhas)
        stem = f"{it['class']}_{os.path.splitext(os.path.basename(it['image_path']))[0]}"
        shutil.copy(it['image_path'], os.path.join(img_dir, stem + '.png'))
        with open(os.path.join(lbl_dir, stem + '.txt'), 'w') as f:
            f.write('\n'.join(linhas))
    return n_lesoes

for nome, subset in [('train', train_items), ('val', val_items), ('test', test_items)]:
    n = materializar(subset, nome)
    print(f'{nome}: {len(subset)} imagens, {n} lesões rotuladas')

## 9. Verificação da estrutura final

In [ ]:
for split in ['train', 'val', 'test']:
    imgs = glob.glob(os.path.join(base_dir, 'images', split, '*.png'))
    lbls = glob.glob(os.path.join(base_dir, 'labels', split, '*.txt'))
    vazios = sum(1 for l in lbls if os.path.getsize(l) == 0)
    print(f'{split:5s}: {len(imgs)} imagens | {len(lbls)} rótulos | {vazios} negativos (normais)')

## 10. Carregamento do YOLO11s (pesos COCO)

In [ ]:
model = YOLO('yolo11s.pt')

## 11. Configuração do dataset (YAML)

In [ ]:
yaml_content = {
    'path': base_dir,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 2,
    'names': ['benigno', 'maligno'],
}
yaml_file_path = os.path.join(base_dir, 'breast_ultrasound.yaml')
with open(yaml_file_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, allow_unicode=True)
print('YAML criado:', yaml_file_path)
print(yaml.dump(yaml_content, allow_unicode=True))

## 12. Treinamento

In [ ]:
trained_model_path = os.path.join(base_dir, 'best_model.pt')

if not os.path.exists(trained_model_path):
    print('Iniciando o treinamento...')
    results = model.train(
        data=yaml_file_path,
        epochs=100,
        imgsz=640,
        batch=16,
        workers=4,
        patience=25,
        seed=SEED,
        optimizer='auto',
        degrees=10.0,     # rotação leve (benéfica em US)
        fliplr=0.5,       # espelhamento horizontal
        flipud=0.0,       # sem espelhamento vertical
        translate=0.1,
        scale=0.3,
        shear=0.0,        # sem cisalhamento
        perspective=0.0,  # sem perspectiva
        project=os.path.join(base_dir, 'runs'),
        name='train',
    )
    best = os.path.join(base_dir, 'runs', 'train', 'weights', 'best.pt')
    shutil.copy(best, trained_model_path)
    print('Modelo salvo em', trained_model_path)
else:
    print('Modelo treinado já existe. Pulando o treinamento.')

## 13. Carregamento do modelo treinado

In [ ]:
model = YOLO(trained_model_path)

## 14. Avaliação no conjunto de teste

`model.val()` com casamento por IoU: mAP@50, mAP@50–95 e P/R/F1 por classe.

In [ ]:
metrics = model.val(split='test', project=os.path.join(base_dir, 'runs'), name='val_test')

resultados = {
    'global': {
        'mAP50': float(metrics.box.map50),
        'mAP50_95': float(metrics.box.map),
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
    },
    'por_classe': {},
}

print('==== Métricas globais (teste) ====')
print(f"mAP@50    : {resultados['global']['mAP50']:.4f}")
print(f"mAP@50-95 : {resultados['global']['mAP50_95']:.4f}")
print(f"Precisão  : {resultados['global']['precision']:.4f}")
print(f"Revocação : {resultados['global']['recall']:.4f}")

print('\n==== Métricas por classe ====')
print(f'{"classe":10s} {"P":>7s} {"R":>7s} {"F1":>7s} {"AP@50":>7s} {"AP@50-95":>9s}')
for i, c in enumerate(metrics.box.ap_class_index):
    p, r, ap50, ap = metrics.box.class_result(i)
    f1 = 2 * p * r / (p + r + 1e-9)
    nome = metrics.names[c]
    resultados['por_classe'][nome] = {
        'P': float(p), 'R': float(r), 'F1': float(f1),
        'AP50': float(ap50), 'AP50_95': float(ap),
    }
    print(f'{nome:10s} {p:7.3f} {r:7.3f} {f1:7.3f} {ap50:7.3f} {ap:9.3f}')

print('\nResultados e gráficos salvos em:', metrics.save_dir)

## 15. Taxa de falsos positivos em imagens normais

In [ ]:
normais_teste = glob.glob(os.path.join(base_dir, 'images', 'test', 'normal_*.png'))
fp = 0
for img_path in normais_teste:
    r = model(img_path, verbose=False)[0]
    if r.boxes is not None and len(r.boxes) > 0:
        fp += 1

total = len(normais_teste)
taxa = (fp / total * 100) if total else 0.0
resultados['falsos_positivos_normais'] = {'total': total, 'com_deteccao': fp, 'taxa_pct': taxa}
print(f'Imagens normais (teste): {total}')
print(f'Com detecção espúria   : {fp} ({taxa:.1f}%)')

## 16. Inferência qualitativa

In [ ]:
from IPython.display import Image, display

def mostrar_exemplos(model, split_dir, prefixo, n=3):
    imgs = sorted(glob.glob(os.path.join(split_dir, f'{prefixo}_*.png')))[:n]
    out_dir = os.path.join(base_dir, 'inference_results')
    os.makedirs(out_dir, exist_ok=True)
    for img_path in imgs:
        r = model(img_path, verbose=False)[0]
        save_path = os.path.join(out_dir, os.path.basename(img_path))
        cv2.imwrite(save_path, r.plot())
        display(Image(filename=save_path))

test_dir = os.path.join(base_dir, 'images', 'test')
print('Exemplos malignos:');  mostrar_exemplos(model, test_dir, 'malignant')
print('Exemplos benignos:');  mostrar_exemplos(model, test_dir, 'benign')
print('Exemplos normais:');   mostrar_exemplos(model, test_dir, 'normal')

## 17. Exportação dos resultados

Gera `resultados_experimentais.json` e `tabela_resultados.md` com todas as métricas.

In [ ]:
export = {
    'ambiente': env_info,
    'seed': SEED,
    'conjuntos': split_counts,
    'modelo': 'YOLO11s (transfer learning COCO)',
    'hiperparametros': {
        'epochs': 100, 'imgsz': 640, 'batch': 16, 'patience': 25,
        'augment': {'degrees': 10.0, 'fliplr': 0.5, 'flipud': 0.0,
                    'translate': 0.1, 'scale': 0.3, 'shear': 0.0, 'perspective': 0.0},
    },
    'resultados': resultados,
}

json_path = os.path.join(base_dir, 'resultados_experimentais.json')
with open(json_path, 'w') as f:
    json.dump(export, f, indent=2, ensure_ascii=False)
print('JSON salvo em', json_path)

# Tabela markdown pronta
linhas = ['# Resultados experimentais\n',
          f"Modelo: {export['modelo']} | Semente: {SEED} | "
          f"GPU: {env_info.get('gpu','?')} | ultralytics {env_info.get('ultralytics','?')}\n",
          '## Métricas globais (teste)\n',
          '| mAP@50 | mAP@50-95 | Precisão | Revocação |',
          '|---|---|---|---|',
          f"| {resultados['global']['mAP50']:.3f} | {resultados['global']['mAP50_95']:.3f} | "
          f"{resultados['global']['precision']:.3f} | {resultados['global']['recall']:.3f} |\n",
          '## Métricas por classe (teste)\n',
          '| Classe | P | R | F1 | AP@50 | AP@50-95 |',
          '|---|---|---|---|---|---|']
for nome, m in resultados['por_classe'].items():
    linhas.append(f"| {nome} | {m['P']:.3f} | {m['R']:.3f} | {m['F1']:.3f} | "
                  f"{m['AP50']:.3f} | {m['AP50_95']:.3f} |")
fp_info = resultados['falsos_positivos_normais']
linhas += ['', '## Falsos positivos em imagens normais\n',
           f"{fp_info['com_deteccao']} de {fp_info['total']} "
           f"({fp_info['taxa_pct']:.1f}%) geraram detecção espúria."]
md_path = os.path.join(base_dir, 'tabela_resultados.md')
with open(md_path, 'w') as f:
    f.write('\n'.join(linhas))
print('Tabela salva em', md_path)
print('\n' + '\n'.join(linhas))